In [1]:
from openai import OpenAI
import json, time
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
client = OpenAI()

def call_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


In [ ]:

PRODUCT_CONFIGS = {

    "saas_crm": {
        "product_name": "CRM Platform",
        "categories"  : ["Billing", "Technical", "Feature Request", "Spam", "Other"],
        "urgency"     : ["High", "Medium", "Low"],
        "sla_hours"   : {"High": 2, "Medium": 8, "Low": 24},
        "examples"    : [
            {
                "subject": "SFDC not syncing",
                "body"   : "Salesforce stopped pulling leads.",
                "output" : "Technical"
            },
            {
                "subject": "Charged wrong amount",
                "body"   : "Billed $299 instead of $99.",
                "output" : "Billing"
            }
        ]
    },

    "fintech_payments": {
        "product_name": "Payment Gateway",
        "categories"  : ["Transaction Failed", "Settlement Delay", "KYC Issue", "Fraud Alert", "Other"],
        "urgency"     : ["Critical", "High", "Medium"],
        "sla_hours"   : {"Critical": 1, "High": 4, "Medium": 12},
        "examples"    : [
            {
                "subject": "UPI payment failed",
                "body"   : "UPI transaction deducted but not credited to merchant.",
                "output" : "Transaction Failed"
            },
            {
                "subject": "KYC rejected again",
                "body"   : "Documents rejected for 3rd time. Aadhaar + PAN submitted.",
                "output" : "KYC Issue"
            }
        ]
    },

    "edtech": {
        "product_name": "EdTech Learning Platform",
        "categories"  : ["Course Access", "Payment", "Technical", "Content Query", "Spam", "Other"],
        "urgency"     : ["High", "Medium", "Low"],
        "sla_hours"   : {"High": 4, "Medium": 12, "Low": 48},
        "examples"    : [
            {
                "subject": "Can't access purchased course",
                "body"   : "Paid for the Python course but can't open any videos.",
                "output" : "Course Access"
            },
            {
                "subject": "Certificate not generated",
                "body"   : "Completed the course 3 days ago. Certificate still pending.",
                "output" : "Technical"
            }
        ]
    }
}


In [ ]:
def build_dynamic_prompt(product_key, subject, body):
    """
    Build a product-specific prompt from config.
    Zero hardcoding — everything comes from PRODUCT_CONFIGS.
    """
    config = PRODUCT_CONFIGS[product_key]

    # Build categories list dynamically
    categories_str = "\n".join([f"  - {cat}" for cat in config['categories']])

    # Build examples block dynamically
    examples_str = ""
    for i, ex in enumerate(config['examples'], 1):
        examples_str += f"""
EXAMPLE {i}:
Subject: {ex['subject']}
Body: {ex['body']}
Output: {ex['output']}
"""

    # Build SLA context
    sla_str = ", ".join([
        f"{k}: {v}hr response"
        for k, v in config['sla_hours'].items()
    ])

    return f"""CRITICAL: Return ONLY the category name.

You are an expert support email classifier for {config['product_name']}.

CATEGORIES (pick exactly one):
{categories_str}

URGENCY SLAs: {sla_str}

EXAMPLES FROM THIS PRODUCT:
{examples_str}

NOW CLASSIFY:
Subject: {subject}
Body: {body}

Return ONLY the category name."""

In [25]:
def classify_dynamic(product_key, email):
    prompt = build_dynamic_prompt(
        product_key,
        email['subject'],
        email['body']
    )
    print(prompt)
    return call_llm(prompt)


In [26]:
test_emails = [
    {"product": "saas_crm",        "subject": "Login broken",           "body": "Can't sign in since morning.",              "expected": "Technical"},
    {"product": "fintech_payments", "subject": "Payment not credited",   "body": "Customer paid but order not confirmed.",     "expected": "Transaction Failed"},
    {"product": "edtech",           "subject": "Video not playing",      "body": "Purchased course but videos won't load.",   "expected": "Course Access"},
]

In [27]:
for item in test_emails:
    result = classify_dynamic(item['product'],item)
    print(result)

CRITICAL: Return ONLY the category name.

You are an expert support email classifier for CRM Platform.

CATEGORIES (pick exactly one):
  - Billing
  - Technical
  - Feature Request
  - Spam
  - Other

URGENCY SLAs: High: 2hr response, Medium: 8hr response, Low: 24hr response

EXAMPLES FROM THIS PRODUCT:

EXAMPLE 1:
Subject: SFDC not syncing
Body: Salesforce stopped pulling leads.
Output: Technical

EXAMPLE 2:
Subject: Charged wrong amount
Body: Billed $299 instead of $99.
Output: Billing


NOW CLASSIFY:
Subject: Login broken
Body: Can't sign in since morning.

Return ONLY the category name.
Technical
CRITICAL: Return ONLY the category name.

You are an expert support email classifier for Payment Gateway.

CATEGORIES (pick exactly one):
  - Transaction Failed
  - Settlement Delay
  - KYC Issue
  - Fraud Alert
  - Other

URGENCY SLAs: Critical: 1hr response, High: 4hr response, Medium: 12hr response

EXAMPLES FROM THIS PRODUCT:

EXAMPLE 1:
Subject: UPI payment failed
Body: UPI transactio

In [30]:
## COT vs without COT
email = {
    "subject": "Can't login ",
    "body"   : "Our entire team can't login.  paid for annual "
               "plan last week. Board demo in 3 hours.entire team cant login",
    "correct": "Billing"
}

subject = email['subject']
body= email['body']

PROMPT_CURRENT = f"""You are an expert support email classifier.

Classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Return ONLY: Category | Confidence (1-10)"""


result = call_llm(PROMPT_CURRENT)

In [31]:
result

'Technical | 9'

In [ ]:
PROMPT_COT_V2 = f"""You are an expert support email classifier
for a B2B SaaS company.

Before classifying, think step by step:
1. What is the surface complaint?
2. What is the ROOT CAUSE behind it?
3. Which team OWNS this problem?
4. What is the business impact?

Important:
- Technical team fixes bugs and crashes
- Billing team fixes payment and account activation issues
- Surface complaint does not decide the category
- ROOT CAUSE decides the category

Then classify into EXACTLY ONE:
- Billing
- Technical
- Feature Request
- Spam
- Other

Subject : {subject}
Body    : {body}

Format EXACTLY like this:
THINKING:
1. Surface complaint: ...
2. Root cause: ...
3. Team that OWNS this: ...
4. Business impact: ...

CATEGORY: ..."""


In [39]:
result = call_llm(PROMPT_COT_V2)
print(result)

THINKING:
1. Surface complaint: Can't login.
2. Root cause: Account activation issue related to the recent payment for the annual plan.
3. Team that OWNS this: Technical team.
4. Business impact: The inability to log in affects the entire team's access to the platform, jeopardizing an important board demo scheduled in 3 hours.

CATEGORY: Technical


In [ ]:
config = PRODUCT_CONFIGS['fintech_payments']

categories_str = ".\n".join([cat for cat in config['categories']])
categories_str



# categories = ['Billing', 'Technical', 'Feature Request', 'Spam', 'Other']

'Transaction Failed.\nSettlement Delay.\nKYC Issue.\nFraud Alert.\nOther'

In [14]:
email = 'I paid my emi 2 times'

In [18]:
prompt_saas = f"""

You are email classfier. classify the below email : {email}
and category should be 
- SPAM
- Billing issue

"""

prompt_fin = f"""

You are email classfier. classify the below email : {email}
and category should be 
- Tx issue
- EMI issue

"""

In [19]:
call_llm(prompt_fin)

'The email can be classified under the category: **EMI issue**.'